# Render Remotion 1080p 30fps di Google Colab (T4 GPU)

Sebelum menjalankan sel di bawah:
1. Pastikan runtime memakai **T4 GPU** (*Runtime → Change runtime type → T4 GPU*).
2. Jika ingin menyimpan aman ke Google Drive, jalankan **Sel 1 (Mount Drive)** lalu **Sel 2 (Setup)**.
3. Jika merender **`SejarahIslam`** di **akun Colab lain**, Anda cukup buka notebook ini di akun ke-2 lalu jalankan **Sel 1** (opsional) → **Sel 5** (Sel 5 sudah otomatis setup & download Chrome jika belum ada), lalu jalankan sel **Download MP4** di bawahnya.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/remotion-out

In [ ]:
%%shell
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak aktif! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai! Siap render."

In [ ]:
%%shell
# (Opsional) Tes cepat 10 detik pertama (frames 0-299) untuk cek kecepatan & tampilan sebelum render penuh
set -e
cd /content/vidio
OUT_DIR="/content/drive/MyDrive/remotion-out"
[ -d /content/drive/MyDrive ] || OUT_DIR="/content/remotion-out"
mkdir -p "$OUT_DIR"

if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  npx remotion render SejarahYunani "$OUT_DIR/test-yunani-10s.mp4" \
    --frames=0-299 \
    --chrome-mode=chrome-for-testing \
    --gl=angle-egl \
    --hardware-acceleration=if-possible \
    --video-bitrate=8M
else
  npx remotion render SejarahYunani "$OUT_DIR/test-yunani-10s.mp4" \
    --frames=0-299 \
    --x264-preset=veryfast \
    --crf=21
fi
ls -lh "$OUT_DIR/test-yunani-10s.mp4"

In [ ]:
%%shell
# Sel 4 - Render Video 1: SejarahYunani (1920x1080 @ 30fps, 24.131 frames)
set -e
cd /content/vidio
OUT_DIR="/content/drive/MyDrive/remotion-out"
[ -d /content/drive/MyDrive ] || OUT_DIR="/content/remotion-out"
mkdir -p "$OUT_DIR"

if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  echo "Mode GPU T4: chrome-for-testing + angle-egl + NVENC"
  npx remotion render SejarahYunani "$OUT_DIR/sejarah-yunani-1080p.mp4" \
    --chrome-mode=chrome-for-testing \
    --gl=angle-egl \
    --hardware-acceleration=if-possible \
    --video-bitrate=8M
else
  echo "Mode CPU: libx264 veryfast"
  npx remotion render SejarahYunani "$OUT_DIR/sejarah-yunani-1080p.mp4" \
    --x264-preset=veryfast \
    --crf=21
fi
ls -lh "$OUT_DIR/sejarah-yunani-1080p.mp4"

In [ ]:
# (Opsional) Download langsung sejarah-yunani-1080p.mp4 ke laptop
from google.colab import files
import os

for p in [
    '/content/drive/MyDrive/remotion-out/sejarah-yunani-1080p.mp4',
    '/content/remotion-out/sejarah-yunani-1080p.mp4',
]:
    if os.path.exists(p):
        print('Mengunduh:', p)
        files.download(p)
        break
else:
    print('File sejarah-yunani-1080p.mp4 belum ditemukan. Jalankan sel render SejarahYunani dulu.')

In [ ]:
%%shell
# Sel 5 - Render Video 2: SejarahIslam (1920x1080 @ 30fps, 27.280 frames)
# Mandiri: jika dijalankan di akun Colab baru tanpa lewat Sel 2, setup akan dijalankan otomatis.
set -e

if [ ! -d /content/vidio/node_modules ]; then
  echo "=== Auto-setup environment untuk akun baru ==="
  DEBIAN_FRONTEND=noninteractive apt-get update -qq
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
    libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
    libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
    libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
    libvulkan1 fonts-liberation
  (DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)
  NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
  if [ "$NODE_MAJOR" -lt 20 ]; then
    curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
    DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
  fi
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
  cd /content/vidio && npm ci
fi

cd /content/vidio
OUT_DIR="/content/drive/MyDrive/remotion-out"
[ -d /content/drive/MyDrive ] || OUT_DIR="/content/remotion-out"
mkdir -p "$OUT_DIR"

if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  echo "Mode GPU T4: chrome-for-testing + angle-egl + NVENC (auto-download Chrome jika belum ada)"
  npx remotion render SejarahIslam "$OUT_DIR/sejarah-islam-1080p.mp4" \
    --chrome-mode=chrome-for-testing \
    --gl=angle-egl \
    --hardware-acceleration=if-possible \
    --video-bitrate=8M
else
  echo "Mode CPU: libx264 veryfast"
  npx remotion render SejarahIslam "$OUT_DIR/sejarah-islam-1080p.mp4" \
    --x264-preset=veryfast \
    --crf=21
fi
ls -lh "$OUT_DIR/sejarah-islam-1080p.mp4"

In [ ]:
# Download langsung sejarah-islam-1080p.mp4 ke laptop
from google.colab import files
import os

for p in [
    '/content/drive/MyDrive/remotion-out/sejarah-islam-1080p.mp4',
    '/content/remotion-out/sejarah-islam-1080p.mp4',
]:
    if os.path.exists(p):
        print('Mengunduh:', p)
        files.download(p)
        break
else:
    print('File sejarah-islam-1080p.mp4 belum ditemukan. Jalankan sel render SejarahIslam dulu.')